# Bias in Bios Concept-QA

Train the reusable Concept-QA module for `LabHC/bias_in_bios`.

Main choices:

- input encoder: `sentence-transformers/all-MiniLM-L6-v2`
- query vocabulary: fixed BIOS concept set
- supervision: hybrid encoder-similarity and rule-based soft targets
- artifact: small Concept-QA checkpoint and optional split-level cache

In [ ]:
# %pip install -e ..
# %pip install datasets sentence-transformers scikit-learn -q

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset

from staq.config import default_paths, detect_device
from staq.models import ConceptNet2
from staq.training import seed_everything

In [ ]:
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
paths = default_paths(repo_root)
paths.ensure_artifact_dirs()

device = detect_device()
random_seed = 13
seed_everything(random_seed)

encoder_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_dim = 384

dataset_name = "LabHC/bias_in_bios"
text_column = "hard_text"
target_column = "profession"
sensitive_column = "gender"

profession_names = [
    "accountant",
    "architect",
    "attorney",
    "chiropractor",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "personal_trainer",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "software_engineer",
    "surgeon",
    "teacher",
    "yoga_teacher",
]

gender_names = ["male", "female"]

{
    "device": str(device),
    "encoder_name": encoder_name,
    "embedding_dim": embedding_dim,
    "dataset_name": dataset_name,
}

In [ ]:
raw_train = load_dataset(dataset_name, split="train")
raw_dev = load_dataset(dataset_name, split="dev")
raw_test = load_dataset(dataset_name, split="test")

{
    "train": len(raw_train),
    "dev": len(raw_dev),
    "test": len(raw_test),
    "columns": raw_train.column_names,
}

In [ ]:
concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
concepts_df = pd.read_csv(concepts_path)

concept_names = concepts_df["concept"].tolist()
concept_texts = concepts_df["prompt"].tolist()
sensitive_concepts = concepts_df.query("kind != 'utility'")["concept"].tolist()
sensitive_mask = torch.tensor(concepts_df["kind"].ne("utility").to_numpy(), dtype=torch.bool)

{
    "num_concepts": len(concepts_df),
    "by_kind": concepts_df["kind"].value_counts().to_dict(),
    "num_sensitive_concepts": int(sensitive_mask.sum()),
}

In [ ]:
encoder = SentenceTransformer(encoder_name, device=str(device))

concept_embeddings = encoder.encode(
    concept_texts,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).to(device)

concept_dictionary = concept_embeddings.T.contiguous()

{
    "concept_embeddings": tuple(concept_embeddings.shape),
    "concept_dictionary": tuple(concept_dictionary.shape),
}

In [ ]:
sample_rows = raw_train.select(range(8))
sample_texts = [row[text_column] for row in sample_rows]

sample_embeddings = encoder.encode(
    sample_texts,
    batch_size=8,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).to(device)

sample_concept_qa_inputs = torch.cat(
    [
        sample_embeddings.unsqueeze(1).repeat(1, len(concept_names), 1),
        concept_embeddings.unsqueeze(0).repeat(len(sample_texts), 1, 1),
    ],
    dim=2,
).reshape(len(sample_texts) * len(concept_names), -1)

{
    "sample_embeddings": tuple(sample_embeddings.shape),
    "concept_qa_input_shape": tuple(sample_concept_qa_inputs.shape),
    "expected_concept_qa_input_dim": embedding_dim * 2,
}

In [ ]:
pd.DataFrame(
    {
        "profession": [profession_names[row[target_column]] for row in sample_rows],
        "gender": [gender_names[row[sensitive_column]] for row in sample_rows],
        "text": sample_texts,
    }
)

In [ ]:
encoding_batch_size = 256
raw_splits = {
    "train": raw_train,
    "dev": raw_dev,
    "test": raw_test,
}


def encode_bios_split(split):
    texts = [row[text_column] for row in split]
    embeddings = encoder.encode(
        texts,
        batch_size=encoding_batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).to(device)
    professions = [profession_names[row[target_column]] for row in split]
    genders = [gender_names[row[sensitive_column]] for row in split]
    return {
        "texts": texts,
        "embeddings": embeddings,
        "similarity_scores": embeddings @ concept_embeddings.T,
        "profession_labels": professions,
        "gender_labels": genders,
        "profession_ids": torch.tensor([row[target_column] for row in split], dtype=torch.long),
        "gender_ids": torch.tensor([row[sensitive_column] for row in split], dtype=torch.long),
    }


encoded_splits = {
    split_name: encode_bios_split(split)
    for split_name, split in raw_splits.items()
}

{
    split_name: {
        "embeddings": tuple(split_data["embeddings"].shape),
        "similarity_scores": tuple(split_data["similarity_scores"].shape),
    }
    for split_name, split_data in encoded_splits.items()
}

In [ ]:
train_similarity_scores = encoded_splits["train"]["similarity_scores"]
concept_medians = train_similarity_scores.median(dim=0).values
concept_stds = train_similarity_scores.std(dim=0).clamp_min(1e-6)
calibration_temperature = 1.5


def calibrate_similarity_scores(similarity_scores):
    return torch.sigmoid(
        (similarity_scores - concept_medians.unsqueeze(0))
        / (calibration_temperature * concept_stds.unsqueeze(0))
    )


rule_target_low = 0.05
rule_target_high = 0.95
rule_concept_patterns = {
    "direct_gender_pronouns": r"\b(he|him|his|she|her|hers)\b",
    "gendered_titles": r"\b(mr|mrs|ms|miss|sir|madam)\.?\b",
    "gendered_family_roles": r"\b(wife|wives|husband|husbands|mother|mothers|father|fathers|daughter|daughters|son|sons)\b",
    "gendered_organization_or_award": r"\b(women'?s|men'?s|female|male|girls?|boys?)\b",
}
compiled_rule_patterns = {
    concept: re.compile(pattern, flags=re.IGNORECASE)
    for concept, pattern in rule_concept_patterns.items()
}


def apply_hybrid_targets(split_data):
    encoder_targets = calibrate_similarity_scores(split_data["similarity_scores"])
    hybrid_targets = encoder_targets.clone()
    rule_hits = {}

    for concept, pattern in compiled_rule_patterns.items():
        concept_index = concept_names.index(concept)
        hits = torch.tensor(
            [bool(pattern.search(text)) for text in split_data["texts"]],
            dtype=torch.float32,
            device=device,
        )
        hybrid_targets[:, concept_index] = rule_target_low + (rule_target_high - rule_target_low) * hits
        rule_hits[concept] = hits.bool().cpu()

    split_data["encoder_soft_targets"] = encoder_targets
    split_data["calibrated_soft_targets"] = hybrid_targets
    split_data["rule_hits"] = rule_hits


for split_data in encoded_splits.values():
    apply_hybrid_targets(split_data)

train_targets = encoded_splits["train"]["calibrated_soft_targets"]
calibrated_summary = pd.DataFrame(
    {
        "concept": concept_names,
        "kind": concepts_df["kind"],
        "mean": train_targets.mean(dim=0).cpu().numpy(),
        "std": train_targets.std(dim=0).cpu().numpy(),
        "p05": torch.quantile(train_targets, 0.05, dim=0).cpu().numpy(),
        "p50": torch.quantile(train_targets, 0.50, dim=0).cpu().numpy(),
        "p95": torch.quantile(train_targets, 0.95, dim=0).cpu().numpy(),
    }
)

calibrated_summary.sort_values("mean", ascending=False)

In [ ]:
rule_target_diagnostics = []
for split_name, split_data in encoded_splits.items():
    for concept in rule_concept_patterns:
        concept_index = concept_names.index(concept)
        hits = split_data["rule_hits"][concept]
        encoder_scores = split_data["encoder_soft_targets"][:, concept_index].detach().cpu()
        target_scores = split_data["calibrated_soft_targets"][:, concept_index].detach().cpu()
        rule_target_diagnostics.append(
            {
                "split": split_name,
                "concept": concept,
                "hit_rate": float(hits.float().mean()),
                "encoder_mean_if_hit": float(encoder_scores[hits].mean()) if bool(hits.any()) else np.nan,
                "encoder_mean_if_miss": float(encoder_scores[~hits].mean()) if bool((~hits).any()) else np.nan,
                "target_mean_if_hit": float(target_scores[hits].mean()) if bool(hits.any()) else np.nan,
                "target_mean_if_miss": float(target_scores[~hits].mean()) if bool((~hits).any()) else np.nan,
            }
        )

pd.DataFrame(rule_target_diagnostics)

In [ ]:
def show_top_calibrated_concepts(row_index, split_name="train", top_k=8):
    split_data = encoded_splits[split_name]
    scores = split_data["calibrated_soft_targets"][row_index].detach().cpu().numpy()
    encoder_scores = split_data["encoder_soft_targets"][row_index].detach().cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    rule_hit_by_concept = {
        concept: bool(hits[row_index])
        for concept, hits in split_data["rule_hits"].items()
    }
    print(f"profession: {split_data['profession_labels'][row_index]}")
    print(f"gender: {split_data['gender_labels'][row_index]}")
    print(split_data["texts"][row_index])
    return pd.DataFrame(
        {
            "concept": [concept_names[idx] for idx in top_indices],
            "kind": [concepts_df.loc[idx, "kind"] for idx in top_indices],
            "target_score": scores[top_indices],
            "encoder_score": encoder_scores[top_indices],
            "rule_hit": [rule_hit_by_concept.get(concept_names[idx], np.nan) for idx in top_indices],
        }
    )


show_top_calibrated_concepts(0)

In [ ]:
def show_high_calibrated_examples(concept, split_name="train", top_k=5):
    split_data = encoded_splits[split_name]
    concept_index = concept_names.index(concept)
    scores = split_data["calibrated_soft_targets"][:, concept_index].detach().cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame(
        {
            "score": scores[top_indices],
            "profession": [split_data["profession_labels"][idx] for idx in top_indices],
            "gender": [split_data["gender_labels"][idx] for idx in top_indices],
            "text": [split_data["texts"][idx] for idx in top_indices],
        }
    )


show_high_calibrated_examples("software_development")

In [ ]:
batch_size = 256

def make_concept_qa_dataset(split_name):
    split_data = encoded_splits[split_name]
    return TensorDataset(
        split_data["embeddings"].cpu(),
        split_data["calibrated_soft_targets"].cpu(),
    )


train_dataset = make_concept_qa_dataset("train")
dev_dataset = make_concept_qa_dataset("dev")
test_dataset = make_concept_qa_dataset("test")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

{
    "train_examples": len(train_dataset),
    "dev_examples": len(dev_dataset),
    "test_examples": len(test_dataset),
    "num_concepts": len(concept_names),
}

In [ ]:
def build_text_concept_qa_inputs(text_embeddings, concept_embeddings):
    batch_size = text_embeddings.size(0)
    num_concepts = concept_embeddings.size(0)
    text_rep = text_embeddings.unsqueeze(1).repeat(1, num_concepts, 1)
    concept_rep = concept_embeddings.unsqueeze(0).repeat(batch_size, 1, 1)
    return torch.cat([text_rep, concept_rep], dim=2).reshape(batch_size * num_concepts, -1)


def tensor_corrcoef(left, right):
    left = left.flatten()
    right = right.flatten()
    left = left - left.mean()
    right = right - right.mean()
    denom = left.norm() * right.norm()
    if float(denom) == 0.0:
        return torch.tensor(float("nan"), device=left.device)
    return (left @ right) / denom


def rank_tensor(values):
    order = values.flatten().argsort()
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(len(order), device=values.device, dtype=torch.float32)
    return ranks.view_as(values)


@torch.no_grad()
def evaluate_concept_qa_soft(model, loader):
    model.eval()
    losses = []
    hard_accuracies = []
    pred_batches = []
    target_batches = []

    for text_batch, target_batch in loader:
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)
        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = model(inputs).view_as(target_batch)
        probs = torch.sigmoid(logits)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)
        preds = (probs >= 0.5).float()
        hard_targets = (target_batch >= 0.5).float()

        losses.append(float(loss.item()))
        hard_accuracies.append(float((preds == hard_targets).float().mean().item()))
        pred_batches.append(probs.detach().cpu())
        target_batches.append(target_batch.detach().cpu())

    all_preds = torch.cat(pred_batches)
    all_targets = torch.cat(target_batches)
    pearson = tensor_corrcoef(all_preds, all_targets)
    spearman = tensor_corrcoef(rank_tensor(all_preds), rank_tensor(all_targets))

    return {
        "loss": float(np.mean(losses)),
        "mae": float((all_preds - all_targets).abs().mean().item()),
        "rmse": float(torch.sqrt(((all_preds - all_targets) ** 2).mean()).item()),
        "pearson": float(pearson.item()),
        "spearman": float(spearman.item()),
        "hard_accuracy": float(np.mean(hard_accuracies)),
    }

In [ ]:
concept_qa_model = ConceptNet2(embed_dims=embedding_dim).to(device)
optimizer = torch.optim.AdamW(concept_qa_model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 5

concept_qa_history = []
for epoch in range(1, num_epochs + 1):
    concept_qa_model.train()
    train_losses = []

    for text_batch, target_batch in train_loader:
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)

        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = concept_qa_model(inputs).view_as(target_batch)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    dev_metrics = evaluate_concept_qa_soft(concept_qa_model, dev_loader)
    concept_qa_history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "dev_loss": dev_metrics["loss"],
            "dev_mae": dev_metrics["mae"],
            "dev_rmse": dev_metrics["rmse"],
            "dev_pearson": dev_metrics["pearson"],
            "dev_spearman": dev_metrics["spearman"],
            "dev_hard_accuracy": dev_metrics["hard_accuracy"],
        }
    )

concept_qa_test_metrics = evaluate_concept_qa_soft(concept_qa_model, test_loader)

pd.DataFrame(concept_qa_history), concept_qa_test_metrics

In [ ]:
@torch.no_grad()
def predict_concept_scores_for_row(row_index, split_name="dev"):
    concept_qa_model.eval()
    split_data = encoded_splits[split_name]
    text_embedding = split_data["embeddings"][row_index : row_index + 1].to(device)
    inputs = build_text_concept_qa_inputs(text_embedding, concept_embeddings)
    logits = concept_qa_model(inputs).view(1, len(concept_names))
    return torch.sigmoid(logits).squeeze(0).cpu().numpy()


row_index = 0
split_name = "dev"
model_scores = predict_concept_scores_for_row(row_index, split_name=split_name)
target_scores = encoded_splits[split_name]["calibrated_soft_targets"][row_index].cpu().numpy()
top_indices = np.argsort(model_scores)[::-1][:8]

pd.DataFrame(
    {
        "concept": [concept_names[idx] for idx in top_indices],
        "kind": [concepts_df.loc[idx, "kind"] for idx in top_indices],
        "model_score": model_scores[top_indices],
        "target_score": target_scores[top_indices],
    }
)

In [ ]:
qa_experiment_name = "bias_in_bios_concept_qa_hybrid_full"
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{qa_experiment_name}.pt"
qa_history_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_history.json"
qa_cache_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_cache.pt"
save_split_cache = False

target_metadata = {
    "target_strategy": "encoder_similarity_for_general_concepts__rules_for_literal_sensitive_concepts",
    "rule_target_low": rule_target_low,
    "rule_target_high": rule_target_high,
    "rule_concept_patterns": rule_concept_patterns,
}

checkpoint_payload = {
    "model_state_dict": concept_qa_model.state_dict(),
    "embedding_dim": embedding_dim,
    "encoder_name": encoder_name,
    "concepts": concepts_df.to_dict(orient="records"),
    "concepts_path": str(concepts_path.relative_to(repo_root)),
    "concept_embeddings": concept_embeddings.detach().cpu(),
    "concept_dictionary": concept_dictionary.detach().cpu(),
    "calibration_medians": concept_medians.detach().cpu(),
    "calibration_stds": concept_stds.detach().cpu(),
    "calibration_temperature": calibration_temperature,
    "target_metadata": target_metadata,
    "test_metrics": concept_qa_test_metrics,
}

torch.save(checkpoint_payload, qa_checkpoint)

if save_split_cache:
    cache_payload = {
        "encoder_name": encoder_name,
        "embedding_dim": embedding_dim,
        "concepts": concepts_df.to_dict(orient="records"),
        "concept_embeddings": concept_embeddings.detach().cpu(),
        "concept_dictionary": concept_dictionary.detach().cpu(),
        "calibration_medians": concept_medians.detach().cpu(),
        "calibration_stds": concept_stds.detach().cpu(),
        "calibration_temperature": calibration_temperature,
        "target_metadata": target_metadata,
        "splits": {
            split_name: {
                "embeddings": split_data["embeddings"].detach().cpu(),
                "similarity_scores": split_data["similarity_scores"].detach().cpu(),
                "encoder_soft_targets": split_data["encoder_soft_targets"].detach().cpu(),
                "calibrated_soft_targets": split_data["calibrated_soft_targets"].detach().cpu(),
                "profession_ids": split_data["profession_ids"].cpu(),
                "gender_ids": split_data["gender_ids"].cpu(),
                "profession_labels": split_data["profession_labels"],
                "gender_labels": split_data["gender_labels"],
            }
            for split_name, split_data in encoded_splits.items()
        },
    }
    torch.save(cache_payload, qa_cache_path)

history_payload = {
    "history": concept_qa_history,
    "test_metrics": concept_qa_test_metrics,
    "target_metadata": target_metadata,
    "rule_target_diagnostics": rule_target_diagnostics,
}
with open(qa_history_path, "w", encoding="utf-8") as handle:
    json.dump(history_payload, handle, indent=2)

{
    "checkpoint": str(qa_checkpoint.relative_to(repo_root)),
    "history": str(qa_history_path.relative_to(repo_root)),
    "cache": str(qa_cache_path.relative_to(repo_root)) if save_split_cache else None,
}